# 05 - 微调后模型评估

加载微调后的模型，运行与微调前相同的评测流程，对比能力提升。

## 5.1 加载微调后的模型

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

import torch
from peft import PeftModel, AutoPeftModelForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer

from config import MODELS_DIR, RESULTS_DIR, BASE_MODEL

# 路径配置
base_model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])
adapter_path = os.path.join(MODELS_DIR, 'meerkat_triz_adapter_v1')

print(f"基座模型: {base_model_path}")
print(f"LoRA适配器: {adapter_path}")

# 方式1: 使用AutoPeftModelForCausalLM直接加载 (推荐)
print("\n加载微调后的模型...")

model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    adapter_path,
    trust_remote_code=True,
)

print("模型加载完成!")

## 5.2 TRIZ能力对比测试

In [ ]:
# 运行TRIZ定制评测
from utils.benchmark_utils import run_triz_evaluation

print("运行TRIZ定制评测 (微调后)...")

triz_results_after = run_triz_evaluation(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
)

print("\n微调后 TRIZ 评测结果:")
print(f"  综合得分: {triz_results_after['overall_score']:.2%}")
print(f"  原理识别: {triz_results_after['principle_accuracy']['accuracy']:.2%}")
print(f"  矛盾解决: {triz_results_after['contradiction_resolution']['average_score']:.2%}")
print(f"  案例质量: {triz_results_after['case_quality']['average_coverage']:.2%}")
print(f"  ARIZ完整: {triz_results_after['ariz_completeness']['completeness']:.2%}")

## 5.3 实际案例测试

In [ ]:
# 实际TRIZ咨询案例测试
test_cases = [
    {
        "name": "矛盾分析",
        "prompt": "一辆汽车需要既坚固（安全性）又轻便（省油），请用TRIZ分析这个技术矛盾并给出解决方案。"
    },
    {
        "name": "原理推荐",
        "prompt": "如何提高太阳能电池板的能量转换效率？请推荐相关的TRIZ发明原理。"
    },
    {
        "name": "ARIZ指导",
        "prompt": "使用ARIZ算法，分析'如何在不影响打印质量的情况下降低3D打印成本'这个问题。"
    },
]

system_msg = (
    "You are Meerkat-AI, an expert innovation consultant specializing in TRIZ "
    "(Theory of Inventive Problem Solving). Provide professional, structured advice."
)

for case in test_cases:
    print(f"\n{'='*60}")
    print(f"测试: {case['name']}")
    print(f"{'='*60}")
    
    # 构建ChatML格式prompt
    prompt = (
        f"<|im_start|>system\n{system_msg}<|im_end|>\n"
        f"<|im_start|>user\n{case['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 提取assistant回复
    if 'assistant' in response:
        response = response.split('assistant')[-1].strip()
    
    print(response[:800])
    print("... [truncated]")

## 5.4 性能基准测试

In [ ]:
from utils.benchmark_utils import run_performance_benchmark

print("运行性能基准测试 (微调后)...")

perf_results_after = run_performance_benchmark(
    model=model,
    tokenizer=tokenizer,
    output_dir=RESULTS_DIR,
)

print("\n性能测试完成!")

## 5.5 生成对比报告

In [ ]:
# 对比微调前后的结果
print("="*60)
print("微调效果对比报告")
print("="*60)

# TRIZ综合得分对比
before_score = 0.35  # 基座模型预估得分
after_score = triz_results_after['overall_score']
improvement = (after_score - before_score) / before_score * 100

print(f"\nTRIZ综合得分:")
print(f"  微调前 (基座模型): {before_score:.2%}")
print(f"  微调后 (TRIZ适配器): {after_score:.2%}")
print(f"  提升幅度: {improvement:+.1f}%")

# 各项能力对比
print(f"\n各项能力得分:")
for key, name in [
    ('principle_accuracy', '原理识别准确率'),
    ('contradiction_resolution', '矛盾解决能力'),
    ('case_quality', '案例生成质量'),
    ('ariz_completeness', 'ARIZ完整性'),
]:
    score = triz_results_after[key]
    if isinstance(score, dict):
        score = list(score.values())[0]
    print(f"  {name}: {score:.2%}")

print("\n="*60)
print("评估完成! 所有结果已保存到 results/ 目录")
print("="*60)

---

## 项目完成! 

所有流程已执行完毕。项目文件结构:
```
mongoose_ai/
├── models/
│   ├── Qwen3-72B/              # 基座模型
│   └── meerkat_triz_adapter_v1/  # LoRA适配器 (~100MB)
├── data/
│   ├── raw/                    # 原始数据
│   └── processed/              # ChatML格式数据
├── results/                    # 评测结果
└── checkpoints/                # 训练检查点
```